In [15]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk -qq
!pip install pyspark==3.5.3 pandas -qq
print("✅ ¡Listo!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ ¡Listo!


In [16]:
import pandas as pd
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# ==== CARGAR TU ARCHIVO EXCEL ====
print("📂 Cargando tus datos...")
df = pd.read_excel("DATOS_VICTIMAS_FISCALIA.xlsx", nrows=30)
print(f"✅ {len(df)} registros cargados")

# ==== INICIAR SPARK ====
spark = SparkSession.builder.appName("Streaming_Simulado").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("✅ Spark iniciado")

# ==== SIMULAR STREAMING ====
print("\n" + "=" * 60)
print("📤 ENVIANDO DATOS EN TIEMPO REAL...")
print("=" * 60)

resultados = []

for i, fila in df.iterrows():
    # Usamos los nombres EXACTOS de tus columnas
    id_victima = str(fila.get("ID VICTIMA ANONIMIZADO", "Sin ID"))
    departamento = str(fila.get("DEPARTAMENTO HECHO ORIGEN", "Sin Dato"))
    delito = str(fila.get("DELITO", "Sin Dato"))

    dato = {
        "ID_VICTIMA": id_victima,
        "DEPARTAMENTO": departamento,
        "DELITO": delito,
        "HORA": datetime.now().strftime("%H:%M:%S")
    }
    resultados.append(dato)
    print(f"✅ [{dato['HORA']}] → {dato['DEPARTAMENTO']} | {dato['DELITO']}")
    time.sleep(1)

print("\n" + "=" * 60)
print("📥 PROCESANDO DATOS RECIBIDOS...")
print("=" * 60)

# Convertir a DataFrame de Spark y analizar
df_spark = spark.createDataFrame(resultados)

print("\n📊 CONTEO POR DEPARTAMENTO:")
df_spark.groupBy("DEPARTAMENTO").agg(count("*").alias("Total")).orderBy("Total", ascending=False).show(truncate=False)

print("\n📊 CONTEO POR TIPO DE DELITO:")
df_spark.groupBy("DELITO").agg(count("*").alias("Total")).orderBy("Total", ascending=False).show(truncate=False)

print("\n🏁 PROCESAMIENTO EN TIEMPO REAL FINALIZADO ✅")

📂 Cargando tus datos...
✅ 30 registros cargados
✅ Spark iniciado

📤 ENVIANDO DATOS EN TIEMPO REAL...
✅ [19:01:09] → Antioquia | Sin Información
✅ [19:01:10] → Valle Del Cauca | Sin Información
✅ [19:01:11] → Valle Del Cauca | Sin Información
✅ [19:01:12] → Meta | Sin Información
✅ [19:01:13] → Meta | Sin Información
✅ [19:01:14] → Meta | Sin Información
✅ [19:01:15] → Meta | Sin Información
✅ [19:01:16] → Meta | Sin Información
✅ [19:01:17] → Meta | Deportacion, Expulsion, Traslado O Desplazamiento Forzado De Poblacion Civil Art. 159 C.P.
✅ [19:01:18] → Meta | Deportacion, Expulsion, Traslado O Desplazamiento Forzado De Poblacion Civil Art. 159 C.P.
✅ [19:01:19] → Meta | Sin Información
✅ [19:01:20] → Meta | Sin Información
✅ [19:01:21] → Meta | Sin Información
✅ [19:01:22] → Meta | Sin Información
✅ [19:01:23] → Meta | Sin Información
✅ [19:01:24] → Meta | Deportacion, Expulsion, Traslado O Desplazamiento Forzado De Poblacion Civil Art. 159 C.P.
✅ [19:01:25] → Meta | Sin Información
✅

In [17]:
print("=" * 60)
print("🌐 ENLACES DE LA INTERFAZ DE SPARK")
print("=" * 60)
print(f"🔗 Jobs:             {spark.sparkContext.uiWebUrl}/jobs")
print(f"🔗 Stages:           {spark.sparkContext.uiWebUrl}/stages")
print(f"🔗 Environment:      {spark.sparkContext.uiWebUrl}/environment")
print(f"🔗 Executors:        {spark.sparkContext.uiWebUrl}/executors")
print("\n💡 Haz clic en cada enlace y toma captura de pantalla")

🌐 ENLACES DE LA INTERFAZ DE SPARK
🔗 Jobs:             http://c0778180d777:4040/jobs
🔗 Stages:           http://c0778180d777:4040/stages
🔗 Environment:      http://c0778180d777:4040/environment
🔗 Executors:        http://c0778180d777:4040/executors

💡 Haz clic en cada enlace y toma captura de pantalla
